# All around table for product analysis

- Add a RFM score to the table
- Create the final table
- Insert data into the final table

## Working query to aggregate the data

In [0]:
%skip
WITH max_date AS (
  SELECT
    MAX(order_date) AS max_date
  FROM data_lakehouse_project.gold.fact_sales
),

base AS (
  SELECT
    p.product_id,
    p.product_name,
    p.sub_category,
    p.category,

    p.product_line,
    p.product_start_date,  

    MIN(s.order_date) AS first_order,
    MAX(s.order_date) AS last_order,
    DATE_DIFF(ANY_VALUE(m.max_date), MAX(s.order_date)) AS days_since_last_order,
    MAX(s.ship_date) AS last_ship,
    MAX(s.due_date) AS last_due,
    AVG(DATE_DIFF(s.ship_date, s.order_date)) AS avg_ship_days,
    AVG(DATE_DIFF(s.due_date, s.ship_date)) AS avg_delivery_days,
    AVG(DATE_DIFF(s.due_date, s.order_date)) AS avg_total_preparation_days,
      
    MIN(s.price) AS price,
    MAX(p.product_cost) AS product_cost,
    SUM(s.sales) AS total_sales,
    SUM(s.quantity) AS total_quantity,
    SUM((s.sales - p.product_cost) * s.quantity) AS total_profit,
    COUNT(DISTINCT s.order_number) AS total_orders,
    ROUND(SUM(s.sales) / NULLIF(COUNT(DISTINCT s.order_number), 0), 2) AS avg_order_value,

    CASE 
      WHEN p.product_end_date IS NULL THEN 'Active' 
      ELSE 'Inactive' 
    END AS status

  FROM data_lakehouse_project.gold.dim_products AS p
  LEFT JOIN data_lakehouse_project.gold.fact_sales AS s
    ON p.product_key = s.product_key
  JOIN max_date AS m
    ON TRUE
  GROUP BY ALL
),

--ABC_score (Product's RFM_score)
rfm_score AS (
  SELECT
      *,

      CASE
        WHEN first_order IS NULL THEN 0
        ELSE NTILE(5) OVER(
          PARTITION BY (days_since_last_order >= 0)
          ORDER BY days_since_last_order DESC)
      END AS recency_score,

      CASE 
        WHEN first_order IS NULL THEN 0
        ELSE NTILE(5) OVER(
          PARTITION BY (total_orders > 0)
          ORDER BY total_orders ASC)
      END AS frequency_score,

      CASE
        WHEN first_order IS NULL THEN 0
        ELSE NTILE(5) OVER(
          PARTITION BY (total_profit > 0)
          ORDER BY total_profit ASC)
      END AS monetary_score
  FROM base AS b
),

abc AS (
  SELECT
    *,
    ROUND((monetary_score + frequency_score + recency_score) /3, 2) rfm_score,
    
    -- Part de chaque produit dans le CA total
    ROUND(total_sales / SUM(total_sales) OVER (), 4) AS revenue_share,
    
    -- CA cumulé trié du plus grand au plus petit
    SUM(total_sales) OVER (
      ORDER BY total_sales DESC
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) / SUM(total_sales) OVER () AS cumulative_revenue_share

  FROM rfm_score
)

SELECT
  * EXCEPT (cumulative_revenue_share, revenue_share),
  ROUND(revenue_share * 100, 2) AS revenue_share,

  CASE
    WHEN cumulative_revenue_share <= 0.80 THEN 'A'  -- top 80% du CA
    WHEN cumulative_revenue_share <= 0.95 THEN 'B'  -- 80% à 95%
    ELSE 'C'                                         -- les 5% restants
  END AS abc_class

FROM abc




## Create the table & Insert data

In [0]:
CREATE OR REPLACE TABLE data_lakehouse_project.platinium.product_360 AS
WITH max_date AS (
  SELECT
    MAX(order_date) AS max_date
  FROM data_lakehouse_project.gold.fact_sales
),

base AS (
  SELECT
    p.product_id,
    p.product_name,
    p.sub_category,
    p.category,

    p.product_line,
    p.product_start_date,  

    MIN(s.order_date) AS first_order,
    MAX(s.order_date) AS last_order,
    DATE_DIFF(ANY_VALUE(m.max_date), MAX(s.order_date)) AS days_since_last_order,
    MAX(s.ship_date) AS last_ship,
    MAX(s.due_date) AS last_due,
    AVG(DATE_DIFF(s.ship_date, s.order_date)) AS avg_ship_days,
    AVG(DATE_DIFF(s.due_date, s.ship_date)) AS avg_delivery_days,
    AVG(DATE_DIFF(s.due_date, s.order_date)) AS avg_total_preparation_days,
      
    MIN(s.price) AS price,
    MAX(p.product_cost) AS product_cost,
    SUM(s.sales) AS total_sales,
    SUM(s.quantity) AS total_quantity,
    SUM((s.sales - p.product_cost) * s.quantity) AS total_profit,
    COUNT(DISTINCT s.order_number) AS total_orders,
    ROUND(SUM(s.sales) / NULLIF(COUNT(DISTINCT s.order_number), 0), 2) AS avg_order_value,

    CASE 
      WHEN p.product_end_date IS NULL THEN 'Active' 
      ELSE 'Inactive' 
    END AS status

  FROM data_lakehouse_project.gold.dim_products AS p
  LEFT JOIN data_lakehouse_project.gold.fact_sales AS s
    ON p.product_key = s.product_key
  JOIN max_date AS m
    ON TRUE
  GROUP BY ALL
),

--ABC_score (Product's RFM_score)
rfm_score AS (
  SELECT
      *,

      CASE
        WHEN first_order IS NULL THEN 0
        ELSE NTILE(5) OVER(
          PARTITION BY (days_since_last_order >= 0)
          ORDER BY days_since_last_order DESC)
      END AS recency_score,

      CASE 
        WHEN first_order IS NULL THEN 0
        ELSE NTILE(5) OVER(
          PARTITION BY (total_orders > 0)
          ORDER BY total_orders ASC)
      END AS frequency_score,

      CASE
        WHEN first_order IS NULL THEN 0
        ELSE NTILE(5) OVER(
          PARTITION BY (total_profit > 0)
          ORDER BY total_profit ASC)
      END AS monetary_score

  FROM base AS b
),

abc AS (
  SELECT
    *,
    ROUND((monetary_score + frequency_score + recency_score) /3, 2) rfm_score,
    
    -- Part de chaque produit dans le CA total
    ROUND(total_sales / SUM(total_sales) OVER (), 4) AS revenue_share,
    
    -- CA cumulé trié du plus grand au plus petit
    SUM(total_sales) OVER (
      ORDER BY total_sales DESC
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) / SUM(total_sales) OVER () AS cumulative_revenue_share

  FROM rfm_score
)

SELECT
  * EXCEPT (cumulative_revenue_share, revenue_share),
  ROUND(revenue_share * 100, 2) AS revenue_share,

  CASE
    WHEN cumulative_revenue_share <= 0.80 THEN 'A'  -- top 80% du CA
    WHEN cumulative_revenue_share <= 0.95 THEN 'B'  -- 80% à 95%
    ELSE 'C'                                         -- les 5% restants
  END AS abc_class

FROM abc
ORDER BY product_id




In [0]:
SELECT 
  *
FROM data_lakehouse_project.platinium.product_360

